# 01. UCI Steel Industry — Data Preparation & Quality Audit

**Dataset Reference:** [UCI Steel Industry Energy Consumption](https://archive.ics.uci.edu/dataset/851/steel+industry+energy+consumption) (DOI: `10.24432/C52G8C`, CC BY 4.0).

This notebook executes the exact data preparation specifications established in `data_preparation.md`:
1. **Raw Verification & Hash Audit:** Enforce SHA-256 checksum and schema checks on `Steel_industry_data.csv`.
2. **Chronological Sorting:** Correct 365 backward interval jumps caused by raw `00:00` positioning.
3. **Quality Gates:** Verify zero missing cells, uniform 15-minute intervals, valid power factor bounds, and calendar alignment.
4. **Target Formulation:** Construct `target_next_60m_kWh` (sum of $t+1 \dots t+4$).
5. **Leak-Free Feature Matrix:** Engineer 15 core features (lags, rolling stats, cyclical time harmonics) available strictly up to observation time $t$.
6. **Chronological 4-Way Splitting:** Train (Jan–Aug), Validation (Sep), Calibration (Oct), and Test (Nov–Dec) with purged boundaries.
7. **Traceability Ledger:** Audit all 35,040 rows via `steel_row_manifest.csv`.
8. **Train Column Profiling & Train-Only EDA:** Generate training baseline profiles and export 5 distribution/correlation figures.
9. **Lazy Sequence Dataset:** Construct `SteelSequenceView` for zero-copy deep learning ingestion.

In [ ]:
# Standard library imports
import hashlib
import json
from pathlib import Path

# Third-party scientific and data processing libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Directory definitions
RAW_DATA_PATH = Path("../data/steel/raw/Steel_industry_data.csv")
PROCESSED_DIR = Path("../data/steel/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Verification constants
EXPECTED_SHA256 = (
    "9B1CEE6F9CB9CD9DF2B95814CA90A9A2FF15B7F5F1FBA0FAE3C643E82072EACC"
)
EXPECTED_ROWS = 35_040
EXPECTED_COLS = 11

In [2]:
def compute_sha256(filepath: Path) -> str:
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    return hasher.hexdigest().upper()

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Raw dataset missing at {RAW_DATA_PATH}. "
        "Please download `Steel_industry_data.csv` from UCI and place it in `data/steel/raw/`."
    )

file_bytes = RAW_DATA_PATH.stat().st_size
file_sha256 = compute_sha256(RAW_DATA_PATH)

print(f"Raw file size: {file_bytes:,} bytes")
print(f"Calculated SHA-256: {file_sha256}")
print(f"Expected SHA-256:   {EXPECTED_SHA256}")

assert file_sha256 == EXPECTED_SHA256, (
    f"SHA-256 hash mismatch! Found {file_sha256}, expected {EXPECTED_SHA256}"
)
print("✔ SHA-256 Checksum verified successfully.")

Raw file size: 2,731,389 bytes
Calculated SHA-256: 9B1CEE6F9CB9CD9DF2B95814CA90A9A2FF15B7F5F1FBA0FAE3C643E82072EACC
Expected SHA-256:   9B1CEE6F9CB9CD9DF2B95814CA90A9A2FF15B7F5F1FBA0FAE3C643E82072EACC
✔ SHA-256 Checksum verified successfully.


In [3]:
# Read CSV with explicit string format
df_raw = pd.read_csv(RAW_DATA_PATH)

print(f"Raw shape: {df_raw.shape}")
assert df_raw.shape == (EXPECTED_ROWS, EXPECTED_COLS), (
    f"Unexpected shape: {df_raw.shape}"
)

# Parse datetime formatted as %d/%m/%Y %H:%M
df_raw["timestamp"] = pd.to_datetime(df_raw["date"], format="%d/%m/%Y %H:%M")

# Count backward jumps in original sequence
backward_jumps = (df_raw["timestamp"].diff() < pd.Timedelta(0)).sum()
print(f"Backward interval jumps in raw file: {backward_jumps} (expected 365 corresponding to end-of-day 00:00)")

# Chronological sorting by timestamp
df_sorted = df_raw.sort_values("timestamp").reset_index(drop=True)

# Re-check monotonicity
is_monotonic = df_sorted["timestamp"].is_monotonic_increasing
print(f"Chronologically monotonic after sorting: {is_monotonic}")
assert is_monotonic, "Dataset must be strictly monotonic after sorting!"

Raw shape: (35040, 11)
Backward interval jumps in raw file: 365 (expected 365 corresponding to end-of-day 00:00)
Chronologically monotonic after sorting: True


In [4]:
# Verify uniform 15-minute intervals
time_deltas = df_sorted["timestamp"].diff().dropna()
unexpected_deltas = (time_deltas != pd.Timedelta(minutes=15)).sum()

# Quality gate metrics
missing_cells = df_sorted.isna().sum().sum()
duplicate_timestamps = df_sorted["timestamp"].duplicated().sum()
negative_active_power = (df_sorted["Usage_kWh"] < 0).sum()
zero_active_power = (df_sorted["Usage_kWh"] == 0).sum()

lagging_pf_out_of_bounds = (~df_sorted["Lagging_Current_Power_Factor"].between(0, 100)).sum()
leading_pf_out_of_bounds = (~df_sorted["Leading_Current_Power_Factor"].between(0, 100)).sum()

# Validate NSM (seconds from midnight)
calculated_nsm = (
    df_sorted["timestamp"].dt.hour * 3600
    + df_sorted["timestamp"].dt.minute * 60
    + df_sorted["timestamp"].dt.second
)
nsm_mismatches = (df_sorted["NSM"] != calculated_nsm).sum()

# Validate Day of Week
day_map = {
    "Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3,
    "Friday": 4, "Saturday": 5, "Sunday": 6
}
weekday_mismatches = (df_sorted["Day_of_week"].map(day_map) != df_sorted["timestamp"].dt.weekday).sum()

quality_gate_results = {
    "total_rows": len(df_sorted),
    "unexpected_intervals": int(unexpected_deltas),
    "missing_cells": int(missing_cells),
    "duplicate_timestamps": int(duplicate_timestamps),
    "negative_usage_records": int(negative_active_power),
    "zero_usage_records": int(zero_active_power),
    "lagging_pf_violations": int(lagging_pf_out_of_bounds),
    "leading_pf_violations": int(leading_pf_out_of_bounds),
    "nsm_mismatches": int(nsm_mismatches),
    "weekday_mismatches": int(weekday_mismatches)
}

print(pd.Series(quality_gate_results))

# Enforce quality assertions
assert unexpected_deltas == 0, "Non-uniform sampling interval detected!"
assert missing_cells == 0, "Missing cells found in raw data!"
assert duplicate_timestamps == 0, "Duplicate timestamps detected!"
assert negative_active_power == 0, "Negative power consumption detected!"
assert zero_active_power == 1, "Expected exactly 1 zero-power record (as documented)!"
assert lagging_pf_out_of_bounds == 0 and leading_pf_out_of_bounds == 0, "Power factor out of range [0, 100]!"
assert nsm_mismatches == 0, "NSM column does not match timestamps!"
assert weekday_mismatches == 0, "Day_of_week column does not match timestamps!"
print("✔ All Quality Gates passed successfully.")

total_rows                35040
unexpected_intervals          0
missing_cells                 0
duplicate_timestamps          0
negative_usage_records        0
zero_usage_records            1
lagging_pf_violations         0
leading_pf_violations         0
nsm_mismatches                0
weekday_mismatches            0
dtype: int64
✔ All Quality Gates passed successfully.


In [5]:
# Define target: total kWh in the next 60 minutes (t+1 ... t+4)
# shift(-k) looks k steps into the future
df = df_sorted.copy()

df["observation_time"] = df["timestamp"]
df["forecast_start"] = df["observation_time"] + pd.Timedelta(minutes=15)
df["forecast_end"] = df["observation_time"] + pd.Timedelta(minutes=60)

df["target_next_60m_kWh"] = (
    df["Usage_kWh"].shift(-1)
    + df["Usage_kWh"].shift(-2)
    + df["Usage_kWh"].shift(-3)
    + df["Usage_kWh"].shift(-4)
)

print("Target variable summary (raw before split trimming):")
print(df["target_next_60m_kWh"].describe())

Target variable summary (raw before split trimming):
count    35036.000000
mean       109.558448
std        126.597467
min          9.860000
25%         12.960000
50%         18.835000
75%        205.425000
max        564.300000
Name: target_next_60m_kWh, dtype: float64


In [6]:
# 1. Autoregressive Lags (past observations only)
df["usage_lag_0"] = df["Usage_kWh"]
df["usage_lag_1"] = df["Usage_kWh"].shift(1)      # 15 minutes prior
df["usage_lag_4"] = df["Usage_kWh"].shift(4)      # 1 hour prior
df["usage_lag_96"] = df["Usage_kWh"].shift(96)    # 24 hours prior
df["usage_lag_672"] = df["Usage_kWh"].shift(672)  # 7 days (1 week) prior

# 2. Trailing Rolling Statistics (strictly closed on past records)
# Using closed='left' on rolling window or rolling on past lags to avoid lookahead
df["usage_sum_last_1h"] = df["Usage_kWh"].rolling(window=4, closed="both").sum()
df["usage_mean_last_4h"] = df["Usage_kWh"].rolling(window=16, closed="both").mean()
df["usage_std_last_4h"] = df["Usage_kWh"].rolling(window=16, closed="both").std(ddof=0)
df["usage_mean_last_24h"] = df["Usage_kWh"].rolling(window=96, closed="both").mean()
df["usage_max_last_24h"] = df["Usage_kWh"].rolling(window=96, closed="both").max()

# 3. Calendar Cyclical Features (derived for forecast_start)
forecast_hour = df["forecast_start"].dt.hour + df["forecast_start"].dt.minute / 60.0
forecast_weekday = df["forecast_start"].dt.weekday

df["forecast_hour_sin"] = np.sin(2 * np.pi * forecast_hour / 24.0)
df["forecast_hour_cos"] = np.cos(2 * np.pi * forecast_hour / 24.0)
df["forecast_weekday_sin"] = np.sin(2 * np.pi * forecast_weekday / 7.0)
df["forecast_weekday_cos"] = np.cos(2 * np.pi * forecast_weekday / 7.0)
df["forecast_is_weekend"] = (forecast_weekday >= 5).astype(int)

# Optional Instantaneous Electrical Features at observation time t
df["lagging_reactive_power_kvarh"] = df["Lagging_Current_Reactive.Power_kVarh"]
df["leading_reactive_power_kvarh"] = df["Leading_Current_Reactive_Power_kVarh"]
df["lagging_power_factor"] = df["Lagging_Current_Power_Factor"]
df["leading_power_factor"] = df["Leading_Current_Power_Factor"]

BASE_FEATURES = [
    "usage_lag_0", "usage_lag_1", "usage_lag_4", "usage_lag_96", "usage_lag_672",
    "usage_sum_last_1h", "usage_mean_last_4h", "usage_std_last_4h",
    "usage_mean_last_24h", "usage_max_last_24h",
    "forecast_hour_sin", "forecast_hour_cos",
    "forecast_weekday_sin", "forecast_weekday_cos",
    "forecast_is_weekend"
]

OPTIONAL_ELECTRICAL_FEATURES = [
    "lagging_reactive_power_kvarh", "leading_reactive_power_kvarh",
    "lagging_power_factor", "leading_power_factor"
]

print(f"Total Base Features engineered: {len(BASE_FEATURES)}")
print(f"Optional Electrical Features: {len(OPTIONAL_ELECTRICAL_FEATURES)}")

Total Base Features engineered: 15
Optional Electrical Features: 4


In [7]:
# Define Calendar Partitions
TRAIN_START = pd.Timestamp("2018-01-01 00:00:00")
VAL_START   = pd.Timestamp("2018-09-01 00:00:00")
CAL_START   = pd.Timestamp("2018-10-01 00:00:00")
TEST_START  = pd.Timestamp("2018-11-01 00:00:00")
YEAR_END    = pd.Timestamp("2018-12-31 23:45:00")

# Initialize Manifest
manifest = pd.DataFrame({
    "row_index": df.index,
    "observation_time": df["observation_time"],
    "forecast_end": df["forecast_end"],
    "split": "unassigned",
    "status": "excluded",
    "reason": "none"
})

# Classify split partitions by observation time
train_mask = (df["observation_time"] >= TRAIN_START) & (df["observation_time"] < VAL_START)
val_mask   = (df["observation_time"] >= VAL_START)   & (df["observation_time"] < CAL_START)
cal_mask   = (df["observation_time"] >= CAL_START)   & (df["observation_time"] < TEST_START)
test_mask  = (df["observation_time"] >= TEST_START)  & (df["observation_time"] <= YEAR_END)

manifest.loc[train_mask, "split"] = "train"
manifest.loc[val_mask, "split"]   = "validation"
manifest.loc[cal_mask, "split"]   = "calibration"
manifest.loc[test_mask, "split"]  = "test"

# Condition 1: Insufficient history (first 672 records of the year lack 7-day lag)
insufficient_hist_mask = df.index < 672
manifest.loc[insufficient_hist_mask, "reason"] = "insufficient_history"

# Condition 2: Trailing future target unavailable (last 4 records of test set)
future_unavail_mask = df["target_next_60m_kWh"].isna()
manifest.loc[future_unavail_mask, "reason"] = "future_label_unavailable"

# Condition 3: Label crosses split boundary (4 steps before boundary)
boundary_cross_train = (df["observation_time"] < VAL_START) & (df["forecast_end"] >= VAL_START)
boundary_cross_val   = (df["observation_time"] < CAL_START) & (df["forecast_end"] >= CAL_START)
boundary_cross_cal   = (df["observation_time"] < TEST_START) & (df["forecast_end"] >= TEST_START)

boundary_cross_mask = boundary_cross_train | boundary_cross_val | boundary_cross_cal
manifest.loc[boundary_cross_mask, "reason"] = "label_crosses_split_boundary"

# Mark included rows
included_mask = (manifest["reason"] == "none")
manifest.loc[included_mask, "status"] = "included"

# Summary audit
print("Row Manifest Accounting:")
print(manifest["reason"].value_counts())
print("Included Rows by Split Partition:")
print(manifest[manifest["status"] == "included"]["split"].value_counts())

# Assert exact counts matching data_preparation.md
assert (manifest["reason"] == "insufficient_history").sum() == 672
assert (manifest["reason"] == "label_crosses_split_boundary").sum() == 12
assert (manifest["reason"] == "future_label_unavailable").sum() == 4
assert (manifest["status"] == "included").sum() == 34_352

split_counts = manifest[manifest["status"] == "included"]["split"].value_counts().to_dict()
assert split_counts["train"] == 22_652
assert split_counts["validation"] == 2_876
assert split_counts["calibration"] == 2_972
assert split_counts["test"] == 5_852
print("✔ Split counts and boundary purges match data_preparation.md exactly.")

Row Manifest Accounting:
reason
none                            34352
insufficient_history              672
label_crosses_split_boundary       12
future_label_unavailable            4
Name: count, dtype: int64
Included Rows by Split Partition:
split
train          22652
test            5852
calibration     2972
validation      2876
Name: count, dtype: int64
✔ Split counts and boundary purges match data_preparation.md exactly.


In [ ]:
# Profile the original 11 columns exclusively on the Train partition
# (All 23,328 observations in Jan-Aug, prior to dropping burn-in)
raw_11_cols = [
    "date", "Usage_kWh", "Lagging_Current_Reactive.Power_kVarh",
    "Leading_Current_Reactive_Power_kVarh", "CO2(tCO2)",
    "Lagging_Current_Power_Factor", "Leading_Current_Power_Factor",
    "NSM", "WeekStatus", "Day_of_week", "Load_Type"
]

train_period_df = df_sorted[df_sorted["timestamp"] < VAL_START][raw_11_cols]
assert len(train_period_df) == 23_328, f"Expected 23,328 train records, got {len(train_period_df)}"

profile_records = []
for col in raw_11_cols:
    s = train_period_df[col]
    rec = {
        "column_name": col,
        "dtype": str(s.dtype),
        "missing_rows": int(s.isna().sum()),
        "unique_values": int(s.nunique())
    }
    if pd.api.types.is_numeric_dtype(s):
        rec["zero_rows"] = int((s == 0).sum())
        rec["min"] = float(s.min())
        rec["median"] = float(s.median())
        rec["p95"] = float(s.quantile(0.95))
        rec["p99"] = float(s.quantile(0.99))
        rec["max"] = float(s.max())
    else:
        rec["zero_rows"] = np.nan
        rec["min"] = np.nan
        rec["median"] = np.nan
        rec["p95"] = np.nan
        rec["p99"] = np.nan
        rec["max"] = np.nan
    profile_records.append(rec)

profile_df = pd.DataFrame(profile_records)
profile_path = PROCESSED_DIR / "steel_train_column_profile.csv"
profile_df.to_csv(profile_path, index=False)
print(f"Saved Train column profile to {profile_path}")
profile_df

Saved Train column profile to ..\data\steel\processed\steel_train_column_profile.csv


,column_name,dtype,missing_rows,unique_values,zero_rows,min,median,p95,p99,max
0,date,object,0,23328,NaN,NaN,NaN,NaN,NaN,NaN
1,Usage_kWh,float64,0,3146,0.0,2.48,4.75,102.0200,124.6003,153.14
2,Lagging_Current_Reactive.Power_kVarh,float64,0,1871,4710.0,0.00,4.90,48.1000,63.6500,96.91
3,Leading_Current_Reactive_Power_kVarh,float64,0,760,16067.0,0.00,0.00,23.8195,26.6000,27.76
4,CO2(tCO2),float64,0,8,13610.0,0.00,0.00,0.0500,0.0600,0.07
5,Lagging_Current_Power_Factor,float64,0,4519,0.0,40.05,88.94,100.0000,100.0000,100.00
6,Leading_Current_Power_Factor,float64,0,2773,0.0,12.50,100.00,100.0000,100.0000,100.00
7,NSM,int64,0,96,243.0,0.00,42750.00,81900.0000,85500.0000,85500.00
8,WeekStatus,object,0,2,NaN,NaN,NaN,NaN,NaN,NaN
9,Day_of_week,object,0,7,NaN,NaN,NaN,NaN,NaN,NaN
